# Descarga y tratamiento del listado de medicamentos de AEMPS



Descargamos el fichero xls de medicamentos de la web de la aemps. 

Tarda bastante, ejecuta solo si quiers actualizar el anterior

In [1]:
import io
import os
import requests
import pandas as pd

URL = "https://listadomedicamentos.aemps.gob.es/medicamentos.xls"
DESTINO = "medicamentos.xls"

# Descarga del fichero Excel en memoria
resp = requests.get(URL, timeout=60)
resp.raise_for_status()
content = resp.content
print(f'Descargados {len(content):,} bytes desde: {URL}')

# Guardar una copia local (útil para reutilizar sin volver a descargar)
with open(DESTINO, 'wb') as f:
    f.write(content)
print(f'Archivo guardado en: {os.path.abspath(DESTINO)}')


Descargados 2,745,421 bytes desde: https://listadomedicamentos.aemps.gob.es/medicamentos.xls
Archivo guardado en: c:\Users\igome\OneDrive - Kerox Technology S.L\Documents\Proyectos\ChatBot Medicamentos\ChatBot_Medicamentos\medicamentos.xls


In [ ]:
# Carga del Excel en un DataFrame
# Nota: Para archivos .xls, pandas usa el motor 'xlrd'. Asegúrate de tenerlo instalado.
try:
    _ = content  # si existe 'content', cargamos desde memoria
    df = pd.read_excel(io.BytesIO(content))
except NameError:
    # si el cuaderno se ejecuta fuera de orden, lee desde el archivo guardado
    df = pd.read_excel("medicamentos.xls")

print(f'DataFrame cargado con forma: {df.shape}')
df.head()


DataFrame cargado con forma: (26623, 15)


,Nº Registro,Medicamento,Laboratorio,Fecha Aut.,Estado,Fecha Estado,Cód. ATC,Principios Activos,Nº P. Activos,¿Comercializado?,¿Triangulo Amarillo?,Observaciones,¿Sustituible?,¿Afecta conducción?,¿Problemas de suministro?
0,06354005IP1,COMPETACT 15 MG/850 MG COMPRIMIDOS RECUBIERTOS...,Takeda Pharma A/S,19/05/2022,Autorizado,20/05/2022,A10BD05,"METFORMINA HIDROCLORURO, PIOGLITAZONA HIDROCLO...",2,NO,NO,Medicamento Sujeto A Prescripción Médica,NaN,NO,NO
1,06356001,EXJADE 125 MG COMPRIMIDOS DISPERSABLES,Novartis Europharm Limited,09/10/2006,Anulado,08/02/2022,V03AC03,DEFERASIROX,1,NO,NO,Diagnóstico Hospitalario,NaN,NO,NO
2,06361001,LUMINITY 150 MICROLITROS/ML GAS Y DISOLVENTE P...,Lantheus Eu Limited,09/10/2006,Autorizado,23/08/2017,V08DA04,PERFLUTRENO,1,SI,NO,Uso Hospitalario,NaN,NO,NO
3,06363006,SPRYCEL 70 MG COMPRIMIDOS RECUBIERTOS CON PELI...,Bristol-Myers Squibb Pharma Eeig,29/11/2006,Autorizado,29/11/2006,L01EA02,DASATINIB MONOHIDRATO,1,SI,NO,Diagnóstico Hospitalario,NaN,SI,NO
4,06363015,SPRYCEL 140 MG COMPRIMIDOS RECUBIERTOS CON PEL...,Bristol-Myers Squibb Pharma Eeig,23/08/2011,Autorizado,23/08/2011,L01EA02,DASATINIB MONOHIDRATO,1,NO,NO,Diagnóstico Hospitalario,NaN,SI,NO


In [6]:
# Información adicional del DataFrame
print('Columnas:', list(df.columns))
df.sample(min(5, len(df)))


Columnas: ['Nº Registro', 'Medicamento', 'Laboratorio', 'Fecha Aut.', 'Estado', 'Fecha Estado', 'Cód. ATC', 'Principios Activos', 'Nº P. Activos', '¿Comercializado?', '¿Triangulo Amarillo?', 'Observaciones', '¿Sustituible?', '¿Afecta conducción?', '¿Problemas de suministro?']


,Nº Registro,Medicamento,Laboratorio,Fecha Aut.,Estado,Fecha Estado,Cód. ATC,Principios Activos,Nº P. Activos,¿Comercializado?,¿Triangulo Amarillo?,Observaciones,¿Sustituible?,¿Afecta conducción?,¿Problemas de suministro?
20711,64346,PENTOXIFILINA SEJMET 400 mg COMPRIMIDOS DE LIB...,Sejmet Pharmaceuticals S.L.,30/10/2001,Suspenso,18/09/2025,C04AD03,PENTOXIFILINA,1,NO,NO,Medicamento Sujeto A Prescripción Médica,NaN,NO,NO
21330,80955,OLMESARTAN/HIDROCLOROTIAZIDA CINFA 40 MG/25 MG...,Laboratorios Cinfa S.A.,09/06/2016,Autorizado,10/06/2016,C09DA08,"HIDROCLOROTIAZIDA, OLMESARTAN MEDOXOMILO",2,SI,NO,Medicamento Sujeto A Prescripción Médica,NaN,NO,NO
7024,40 400-95-96,ASPIRINA COMPRIMIDOS,Bayer Hellas A.E.,09/09/2002,Autorizado,09/09/2002,N02BA01,ACETILSALICILICO ACIDO,1,SI,NO,Sin Receta,NaN,NO,NO
18565,67004,RISPERIDONA KERN PHARMA 6 mg COMPRIMIDOS RECUB...,Kern Pharma S.L.,05/09/2005,Autorizado,05/09/2005,N05AX08,RISPERIDONA,1,SI,NO,Medicamento Sujeto A Prescripción Médica,NaN,SI,NO
5265,71225,PANTOPRAZOL PENSAVITAL 20 mg COMPRIMIDOS GASTR...,Towa Pharmaceutical S.A.,20/07/2009,Autorizado,11/02/2022,A02BC02,PANTOPRAZOL SODICO,1,SI,NO,Sin Receta,NaN,NO,NO


In [10]:
# Crear columna 'nombre' a partir de 'Medicamento'
def _extraer_nombre(valor):
    if not isinstance(valor, str):
        return None
    partes = valor.strip().split()
    if not partes:
        return None
    if partes[0].upper() == 'ACIDO':
        return ' '.join(partes[:2]) if len(partes) >= 2 else 'ACIDO'
    return partes[0]

df['nombre'] = df['Medicamento'].apply(_extraer_nombre)
df[['Medicamento','nombre']].sample(10)

df.to_csv('medicamentos.csv')


In [14]:
prueba_medicamentos=df.iloc[0:10,:]
prueba_medicamentos

,Nº Registro,Medicamento,Laboratorio,Fecha Aut.,Estado,Fecha Estado,Cód. ATC,Principios Activos,Nº P. Activos,¿Comercializado?,¿Triangulo Amarillo?,Observaciones,¿Sustituible?,¿Afecta conducción?,¿Problemas de suministro?,nombre
0,06354005IP1,COMPETACT 15 MG/850 MG COMPRIMIDOS RECUBIERTOS...,Takeda Pharma A/S,19/05/2022,Autorizado,20/05/2022,A10BD05,"METFORMINA HIDROCLORURO, PIOGLITAZONA HIDROCLO...",2,NO,NO,Medicamento Sujeto A Prescripción Médica,NaN,NO,NO,COMPETACT
1,06356001,EXJADE 125 MG COMPRIMIDOS DISPERSABLES,Novartis Europharm Limited,09/10/2006,Anulado,08/02/2022,V03AC03,DEFERASIROX,1,NO,NO,Diagnóstico Hospitalario,NaN,NO,NO,EXJADE
2,06361001,LUMINITY 150 MICROLITROS/ML GAS Y DISOLVENTE P...,Lantheus Eu Limited,09/10/2006,Autorizado,23/08/2017,V08DA04,PERFLUTRENO,1,SI,NO,Uso Hospitalario,NaN,NO,NO,LUMINITY
3,06363006,SPRYCEL 70 MG COMPRIMIDOS RECUBIERTOS CON PELI...,Bristol-Myers Squibb Pharma Eeig,29/11/2006,Autorizado,29/11/2006,L01EA02,DASATINIB MONOHIDRATO,1,SI,NO,Diagnóstico Hospitalario,NaN,SI,NO,SPRYCEL
4,06363015,SPRYCEL 140 MG COMPRIMIDOS RECUBIERTOS CON PEL...,Bristol-Myers Squibb Pharma Eeig,23/08/2011,Autorizado,23/08/2011,L01EA02,DASATINIB MONOHIDRATO,1,NO,NO,Diagnóstico Hospitalario,NaN,SI,NO,SPRYCEL
5,06364007IP,ADROVANCE 70 MG/5.600 UI COMPRIMIDOS,Organon N.V.,12/07/2024,Autorizado,13/07/2024,M05BB03,"COLECALCIFEROL, ALENDRONATO SODIO TRIHIDRATO",2,SI,NO,Medicamento Sujeto A Prescripción Médica,NaN,NO,NO,ADROVANCE
6,06366018,TANDEMACT 30 mg/2 mg COMPRIMIDOS,Cheplapharm Arzneimittel Gmbh,29/09/2009,Autorizado,29/09/2009,A10BD06,"PIOGLITAZONA, GLIMEPIRIDA",2,SI,NO,Medicamento Sujeto A Prescripción Médica,NaN,SI,NO,TANDEMACT
7,06367002,DIACOMIT 250 mg CAPSULAS DURAS,Biocodex,12/11/2009,Autorizado,12/11/2009,N03AX17,ESTIRIPENTOL,1,SI,NO,Diagnóstico Hospitalario,NaN,NO,NO,DIACOMIT
8,06367008,DIACOMIT 250 mg POLVO PARA SUSPENSION ORAL,Biocodex,12/11/2009,Autorizado,12/11/2009,N03AX17,ESTIRIPENTOL,1,SI,NO,Diagnóstico Hospitalario,NaN,NO,NO,DIACOMIT
9,06380003,PREZISTA 400 mg COMPRIMIDOS RECUBIERTOS CON PE...,Janssen-Cilag International N.V,16/02/2009,Autorizado,16/02/2009,J05AE10,DARUNAVIR ETANOLATO,1,SI,NO,Diagnóstico Hospitalario,NaN,NO,NO,PREZISTA
